import all the datasets 


In [31]:
import pandas as pd

import numpy as np
import pickle, json
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression  # instead of Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, LeaveOneOut
from sklearn.metrics import r2_score, mean_absolute_error

DATA IMPOETING

In [32]:
import pandas as pd


## MESSAGE FOR TEAMMATES ==> CHANGE THIS ON OWN MACHINE 
base_path = '~/Desktop/SCHOOL/SEM2/Advanced AI/BUSit week/Attendance AI/Project 2 - Matchday Attendance Prediction/Data/'

df_match = pd.read_csv(base_path + 'gold_match.csv')
df_trends = pd.read_csv(base_path + 'gold_google_trends_daily.csv')
df_tickets = pd.read_csv(base_path + 'gold_match_tickets.csv')
df_context = pd.read_csv(base_path + 'gold_match_context.csv')
df_goals = pd.read_csv(base_path + 'gold_match_goals.csv')
df_articles = pd.read_csv(base_path + 'gold_belga_press_articles.csv', 
                          escapechar='\\', on_bad_lines='skip')


In [33]:
print(df_match.shape)
print(df_trends.shape)
print(df_tickets.shape)
print(df_context.shape)
print(df_goals.shape)
print(df_articles.shape)

(142, 27)
(1351, 3)
(71, 10)
(142, 35)
(372, 18)
(49683, 8)


DATA CLEANING

In [34]:
# ── gold_match ─────────────────────────────────────────────────────────────
df_match['match_date'] = pd.to_datetime(df_match['match_date'])
df_match['is_home_match'] = df_match['is_home_match'].astype(bool)
df_match['tickets_scanned'] = pd.to_numeric(df_match['tickets_scanned'], errors='coerce')
df_match.drop_duplicates(subset='match_id', inplace=True)

# ── gold_google_trends_daily ───────────────────────────────────────────────
df_trends['date'] = pd.to_datetime(df_trends['date'])
df_trends['ohl_interest'] = pd.to_numeric(df_trends['ohl_interest'], errors='coerce')
df_trends.drop_duplicates(subset='date', keep='first', inplace=True)

# ── gold_match_tickets ─────────────────────────────────────────────────────
df_tickets.drop_duplicates(subset='match_id', inplace=True)
# Convert all numeric columns
ticket_cols = df_tickets.columns.drop('match_id')
df_tickets[ticket_cols] = df_tickets[ticket_cols].apply(pd.to_numeric, errors='coerce')

# ── gold_match_context ─────────────────────────────────────────────────────
df_context['match_date'] = pd.to_datetime(df_context['match_date'])
df_context['has_promotion'] = df_context['has_promotion'].astype(bool)
df_context['is_weekend'] = df_context['is_weekend'].astype(bool)
df_context['is_midweek'] = df_context['is_midweek'].astype(bool)
df_context['is_public_holiday'] = df_context['is_public_holiday'].astype(bool)
df_context['is_school_holiday_flanders'] = df_context['is_school_holiday_flanders'].astype(bool)
df_context.drop_duplicates(subset='match_id', inplace=True)

# ── gold_match_goals ───────────────────────────────────────────────────────
df_goals['match_date'] = pd.to_datetime(df_goals['match_date'])
df_goals['is_ohl_goal'] = df_goals['is_ohl_goal'].astype(bool)
df_goals.drop_duplicates(subset='opta_event_id', inplace=True)

# ── gold_belga_press_articles ──────────────────────────────────────────────
df_articles['date'] = pd.to_datetime(df_articles['date'], errors='coerce')
df_articles['article_id'] = pd.to_numeric(df_articles['article_id'], errors='coerce')
df_articles['days_to_match'] = pd.to_numeric(df_articles['days_to_match'], errors='coerce')
df_articles.dropna(subset=['article_id'], inplace=True)
df_articles.drop_duplicates(subset='article_id', inplace=True)
# Strip whitespace from text columns
for col in ['title', 'lead', 'source', 'type']:
    df_articles[col] = df_articles[col].astype(str).str.strip()

# ── Summary ────────────────────────────────────────────────────────────────
for name, df in [('match', df_match), ('trends', df_trends), ('tickets', df_tickets),
                 ('context', df_context), ('goals', df_goals), ('articles', df_articles)]:
    print(f"df_{name}: {df.shape} | nulls: {df.isnull().sum().sum()}")

df_match: (142, 27) | nulls: 66
df_trends: (1350, 3) | nulls: 1208
df_tickets: (71, 10) | nulls: 0
df_context: (142, 35) | nulls: 493
df_goals: (372, 18) | nulls: 491
df_articles: (49683, 8) | nulls: 81810


Checking DATAFRAME for NONE VALUES 
THE OUTPUT OF THIS CODE WILL RESULT IN A LOG FILE 


In [35]:
import os
from datetime import datetime

log_lines = []
log_lines.append(f"=== NA Report — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ===\n")

dataframes = {
    'gold_match': df_match,
    'gold_google_trends_daily': df_trends,
    'gold_match_tickets': df_tickets,
    'gold_match_context': df_context,
    'gold_match_goals': df_goals,
    'gold_belga_press_articles': df_articles,
}

for name, df in dataframes.items():
    na_counts = df.isnull().sum()
    na_cols = na_counts[na_counts > 0]
    total_na = na_cols.sum()
    total_rows = len(df)

    log_lines.append(f"── {name} ({total_rows} rows) ──────────────────────")
    if na_cols.empty:
        log_lines.append("  ✅ No missing values\n")
    else:
        log_lines.append(f"  ⚠️  Total NAs: {total_na}")
        for col, count in na_cols.items():
            pct = (count / total_rows) * 100
            log_lines.append(f"  - {col}: {count} missing ({pct:.1f}%)")
        log_lines.append("")

log_path = os.path.expanduser(base_path + 'na_report.log')
with open(log_path, 'w') as f:
    f.write('\n'.join(log_lines))

print('\n'.join(log_lines))
print(f"\nLog saved to: {log_path}")

=== NA Report — 2026-03-23 14:44:21 ===

── gold_match (142 rows) ──────────────────────
  ⚠️  Total NAs: 66
  - last_result_vs_opponent: 4 missing (2.8%)
  - tickets_scanned: 62 missing (43.7%)

── gold_google_trends_daily (1350 rows) ──────────────────────
  ⚠️  Total NAs: 1208
  - match_id: 1208 missing (89.5%)

── gold_match_tickets (71 rows) ──────────────────────
  ✅ No missing values

── gold_match_context (142 rows) ──────────────────────
  ⚠️  Total NAs: 493
  - promotion_names: 116 missing (81.7%)
  - public_holiday_name: 138 missing (97.2%)
  - school_holiday_name: 100 missing (70.4%)
  - campaign_motto: 139 missing (97.9%)

── gold_match_goals (372 rows) ──────────────────────
  ⚠️  Total NAs: 491
  - assist_player_name: 125 missing (33.6%)
  - var_reviewed: 366 missing (98.4%)

── gold_belga_press_articles (49683 rows) ──────────────────────
  ⚠️  Total NAs: 81810
  - match_id: 40905 missing (82.3%)
  - days_to_match: 40905 missing (82.3%)


Log saved to: /Users/nachatissa

Pre-Processing

In [37]:
# ── Part 2 — Pre-Processing ──────────────────────────────────────────────────
df_home = df_match[df_match['is_home_match'] == True].copy()

# Derive kickoff_hour right here from df_home
df_home['kickoff_hour'] = pd.to_datetime(
    df_home['kickoff_time_local'], format='%H:%M:%S'
).dt.hour

df_full2 = df_home.merge(
    df_tickets[['match_id', 'tickets_sold_total']], on='match_id', how='inner'
).merge(
    df_context, on='match_id', how='left'
)

# Confirm it's there
print("kickoff_hour" in df_full2.columns)        # must print True
print(df_full2['kickoff_hour'].head())

# Engineer opponent avg
df_full2['opponent_avg_attendance'] = df_full2.groupby('away_team')['tickets_sold_total'].transform('mean')
global_avg = df_full2['tickets_sold_total'].mean()

feats = ['opponent_avg_attendance', 'academic_week', 'matchday', 'weather_rain_mm', 'kickoff_hour']

df_clean = df_full2[feats + ['tickets_sold_total', 'away_team', 'match_date_x']].copy()
df_clean['match_date_x'] = pd.to_datetime(df_clean['match_date_x'])
df_clean = df_clean.apply(lambda col: pd.to_numeric(col, errors='ignore'))
df_clean = df_clean.dropna(subset=feats + ['tickets_sold_total'])
df_clean = df_clean.sort_values('match_date_x').reset_index(drop=True)

split_idx = int(len(df_clean) * 0.8)
df_train  = df_clean.iloc[:split_idx].copy()
df_test   = df_clean.iloc[split_idx:].copy()

opp_avg_train = df_train.groupby('away_team')['tickets_sold_total'].mean()
global_avg    = df_train['tickets_sold_total'].mean()

df_train['opponent_avg_attendance'] = df_train['away_team'].map(opp_avg_train)
df_test['opponent_avg_attendance']  = df_test['away_team'].map(opp_avg_train).fillna(global_avg)
df_train = df_train.dropna(subset=['opponent_avg_attendance'])

print(f"Train: {len(df_train)} | Test: {len(df_test)}")

True
0    18
1    18
2    18
3    20
4    16
Name: kickoff_hour, dtype: int32
Train: 56 | Test: 15


Training

In [38]:
X_train = df_train[feats].values
y_train = df_train['tickets_sold_total'].values
X_test  = df_test[feats].values
y_test  = df_test['tickets_sold_total'].values

# Fit scaler on train only, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Train Linear Regression model
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train_scaled, y_train)

LinearRegression()

Evaluation & Save

In [39]:
y_pred_train = model.predict(X_train_scaled)  # model, not LinearRegression
y_pred_test  = model.predict(X_test_scaled)   # model, not LinearRegression
std_resid    = (y_train - y_pred_train).std()

print("=== TRAINING SET ===")
print(f"  R²:   {r2_score(y_train, y_pred_train):.3f}")
print(f"  MAE:  {mean_absolute_error(y_train, y_pred_train):.0f} tickets")

print("\n=== TEST SET ===")
print(f"  R²:   {r2_score(y_test, y_pred_test):.3f}")
print(f"  MAE:  {mean_absolute_error(y_test, y_pred_test):.0f} tickets")

gap = mean_absolute_error(y_test, y_pred_test) - mean_absolute_error(y_train, y_pred_train)
print(f"\nOverfit gap: {gap:.0f} tickets", "✅" if gap < 200 else ("⚠️ Moderate" if gap < 500 else "❌ High"))

loo_mae = -cross_val_score(model, X_train_scaled, y_train, cv=LeaveOneOut(), scoring='neg_mean_absolute_error')  # model, not ridge
print(f"LOO CV MAE:  {loo_mae.mean():.0f} tickets")

# Save
with open('model.pkl', 'wb') as f:
    pickle.dump({'model': model, 'scaler': scaler, 'std': std_resid, 'features': feats}, f)  # model, not LinearRegression

opponent_lookup = opp_avg_train.round(0).to_dict()
opponent_lookup['__global_avg__'] = global_avg
with open('opponent_lookup.json', 'w') as f:
    json.dump(opponent_lookup, f, indent=2)

print("\n✅ model.pkl and opponent_lookup.json saved.")

=== TRAINING SET ===
  R²:   0.374
  MAE:  765 tickets

=== TEST SET ===
  R²:   -0.049
  MAE:  743 tickets

Overfit gap: -21 tickets ✅
LOO CV MAE:  860 tickets

✅ model.pkl and opponent_lookup.json saved.
